In [23]:
with open("tiny.txt", "r") as f:
    text = []
    while True:
        s = f.readline().strip()
        if(s == "<endOfFile>"):
            break
        text.append(s)
text = " ".join(text)
print(text[:200])

First Citizen: Before we proceed any further, hear me speak.  All: Speak, speak.  First Citizen: You are all resolved rather to die than to famish?  All: Resolved. resolved.  First Citizen: First, you


In [24]:
import regex as re
pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

In [25]:
words = re.findall(pat, text)
print(len(set(words)))

14035


In [26]:
byte = text.encode("utf-8")

In [27]:
byte = list(map(int, byte))

In [28]:
def findPairs(text):
    pairs = {}
    i=0
    while(i < len(text)-1):
        pair = (text[i] ,text[i+1])
        pairs[pair] = pairs.get(pair, 0) + 1
        i+=1
    return pairs

In [29]:
def replace(max_pair, ord_text, new_token):
    j=0
    i=0
    temp = []
    while(i < len(ord_text)):
        if(i < len(ord_text)-1 and ord_text[i] == max_pair[0] and ord_text[i+1] == max_pair[1]):
            temp.append(new_token)
            i += 2
            continue
            
        temp.append(ord_text[i])
        i += 1
    return temp

In [33]:
num_merges = 5
merges = {}
byte = text.encode("utf-8")
for i in range(num_merges):
    pairs = findPairs(byte)
    max_pair = max(pairs, key = lambda x: pairs[x])
    if(pairs[max_pair] == 1):
            break
    new_token = 256 + i
    merges[max_pair] = new_token
    byte = replace(max_pair, byte, new_token)
    
    

In [14]:
vocab = {idx: bytes([idx]) for idx in range (256)}

for (p1, p2), idx in merges.items():
    vocab[idx] = vocab[p1] + vocab[p2]

In [15]:
from termcolor import colored
def decode(ids):
    c=0
    for i in ids:
        s = vocab[i]
        s = s.decode("utf-8", errors="replace")
        color = "on_red" if(c%2) else "on_yellow"
        print(colored(s,"black", color), end="")
        c += 1

In [17]:
def encode(text):
    i=0
    tokens = list(text.encode("utf-8"))
    while True:
        i+=1
        pairs = findPairs(tokens)
        index = float("inf")
        to_merge = float("inf")
        for pair, _ in pairs.items():
            cur_index = merges.get(pair, float("inf"))
            if(cur_index < index):
                index = cur_index
                to_merge = pair
        
        if to_merge not in merges:
            break
        tokens = replace(to_merge, tokens, index)
    return tokens

In [18]:
def encodeV2(text):
    words = re.findall(pat, text)
    parts = []
    for text in words:
        tokens = list(text.encode("utf-8"))
        while True:
            pairs = findPairs(tokens)
            index = float("inf")
            to_merge = float("inf")
            for pair, _ in pairs.items():
                cur_index = merges.get(pair, float("inf"))
                if(cur_index < index):
                    index = cur_index
                    to_merge = pair
            
            if to_merge not in merges:
                parts.extend(tokens)
                break
            tokens = replace(to_merge, tokens, index)
    return parts

In [19]:
test = "Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'—words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!"

In [20]:
tokens = encodeV2(test)
tokens2 = encode(test)
decode(tokens)
print()
decode(tokens2)

Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'���words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!
Hello! Welcome to the world of NLP. Tokenization is the first step: breaking down text into 'tokens'���words, phrases, or symbols. For example, the sentence 'I have 2 apples and 3 bananas!' contains numbers (2, 3), punctuation (!, '), and words. Proper tokenization should handle abbreviations (e.g., U.S.A.), contractions (don't, it's), and special characters like #hashtags or @mentions. Can your tokenizer handle this? Let's test it out!